# Boston House Price Prediction
### Exploratory Data Analysis · Feature Scaling · Regression

This notebook explores the Boston housing dataset and builds two regression models: a standard Linear Regression baseline and a Polynomial Regression model.

**What we'll do**
- Inspect the dataset and check data quality.
- Explore distributions, outliers, and relationships between features.
- Compare three common scaling methods on `INDUS`.
- Train and evaluate regression models using a held-out test set.

> **Dataset note:** `MEDV` is the target column in this CSV. The Boston Housing dataset has known historical and ethical limitations; treat this exercise as a learning example, not as a model for real-world housing decisions.

## 1. Setup
Import the libraries used throughout the notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", None)

## 2. Load the data

On Kaggle, the dataset is available at the input path below. If you're running this notebook locally, place `boston.csv` in a `data/` folder.

In [ ]:
from pathlib import Path

kaggle_path = Path("/kaggle/input/datasets/fedesoriano/the-boston-houseprice-data/boston.csv")
local_path = Path("data/boston.csv")

if kaggle_path.exists():
    data_path = kaggle_path
elif local_path.exists():
    data_path = local_path
else:
    raise FileNotFoundError(
        "Couldn't find boston.csv. Attach the Kaggle dataset or place the file at data/boston.csv."
    )

data = pd.read_csv(data_path)
print(f"Loaded: {data_path}")
print(f"Rows: {data.shape[0]:,} | Columns: {data.shape[1]}")

## 3. First look
A quick preview helps confirm that the file loaded as expected.

In [ ]:
data.head()

In [ ]:
data.tail()

### Dataset structure

In [ ]:
data.info()

### Summary statistics
Numeric summaries provide a first sense of the feature ranges and spread.

In [ ]:
data.describe().T

### Missing values
Check the count of missing values in each column before modeling.

In [ ]:
missing_values = data.isna().sum().sort_values(ascending=False)
missing_values

## 4. Exploratory data analysis
### Feature distributions and outliers
The boxplot gives a compact view of the spread and potential outliers in `CRIM` (per-capita crime rate).

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(x=data["CRIM"])
plt.title("Distribution of CRIM")
plt.xlabel("Per-capita crime rate")
plt.tight_layout()
plt.show()

### Correlation between variables
Correlation describes linear association. It does not establish cause and effect, and a strong negative correlation alone is not a reason to remove a feature.

In [ ]:
corr = data.corr(numeric_only=True)

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="YlGnBu", annot=True, fmt=".2f", square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

### Relationship between average rooms and house value
`RM` represents average number of rooms; `MEDV` is the median value target in this dataset.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=data, x="RM", y="MEDV", alpha=0.75)
plt.title("Average Rooms vs. Median House Value")
plt.xlabel("Average number of rooms (RM)")
plt.ylabel("Median house value (MEDV)")
plt.tight_layout()
plt.show()

### Distribution of `INDUS`
`INDUS` is the proportion of non-retail business acres per town. A KDE curve is a smoothed view of its distribution.

In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=data, x="INDUS", fill=True)
plt.title("Distribution of INDUS")
plt.xlabel("INDUS")
plt.tight_layout()
plt.show()

## 5. Feature scaling

Scaling changes the numeric range of a feature; it does not change which observations are present. Here we compare:
- **StandardScaler:** centers around 0 and scales to unit standard deviation.
- **MinMaxScaler:** maps values to the 0–1 range.
- **RobustScaler:** centers using the median and scales by the interquartile range.

This section is a demonstration on `INDUS`. For predictive modeling, fit scalers on training data only and then transform the test data.

In [ ]:
indus = data[["INDUS"]]

scaled_indus = {
    "StandardScaler": StandardScaler().fit_transform(indus).ravel(),
    "MinMaxScaler": MinMaxScaler().fit_transform(indus).ravel(),
    "RobustScaler": RobustScaler().fit_transform(indus).ravel(),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, values) in zip(axes, scaled_indus.items()):
    sns.kdeplot(x=values, fill=True, ax=ax)
    ax.set_title(name)
    ax.set_xlabel("Scaled INDUS")
plt.tight_layout()
plt.show()

## 6. Prepare features and target

We use `MEDV` as the target (`y`) and the remaining columns as input features (`X`). A fixed random seed makes the train/test split reproducible.

In [ ]:
X = data.drop(columns="MEDV")
y = data["MEDV"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training rows: {X_train.shape[0]}")
print(f"Testing rows:  {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

## 7. Linear Regression baseline
The baseline model learns a linear relationship between the input features and the target.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)

### Evaluate the baseline
We report Mean Absolute Error (MAE), Mean Squared Error (MSE), and R². Lower MAE/MSE and a higher R² generally indicate better performance on this test split.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "R²": r2_score(y_true, y_pred),
    }

linear_metrics = regression_metrics(y_test, y_pred_linear)
pd.Series(linear_metrics, name="Linear Regression")

### Actual vs. predicted values
Points near the diagonal indicate predictions close to the observed values.

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=y_pred_linear, alpha=0.75)
limits = [min(y_test.min(), y_pred_linear.min()), max(y_test.max(), y_pred_linear.max())]
plt.plot(limits, limits, linestyle="--", color="black", label="Perfect prediction")
plt.xlabel("Actual MEDV")
plt.ylabel("Predicted MEDV")
plt.title("Linear Regression: Actual vs. Predicted")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Polynomial Regression

Polynomial features allow the model to represent nonlinear relationships. Degree 2 adds squared terms and pairwise interactions.

The transformation is learned from the training data; the test data receives only `transform()`. Polynomial expansion can create many features, so this model is included as a comparison rather than assumed to be better.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

print(f"Features before expansion: {X_train.shape[1]}")
print(f"Features after expansion:  {X_train_poly.shape[1]}")

In [ ]:
poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

y_pred_poly = poly_model.predict(X_test_poly)

### Evaluate Polynomial Regression

In [ ]:
poly_metrics = regression_metrics(y_test, y_pred_poly)
pd.Series(poly_metrics, name="Polynomial Regression (degree 2)")

## 9. Compare models
The comparison uses the same test observations for both models.

In [ ]:
comparison = pd.DataFrame(
    [linear_metrics, poly_metrics],
    index=["Linear Regression", "Polynomial Regression (degree 2)"]
)
comparison

In [ ]:
comparison[["MAE", "MSE"]].plot(kind="bar", figsize=(9, 5))
plt.title("Error Metric Comparison")
plt.ylabel("Error (lower is better)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Takeaways

- EDA helps identify data quality issues, distributions, and relationships before modeling.
- Scaling is useful for many algorithms, though ordinary Linear Regression does not require it in the same way distance- or gradient-based methods often do.
- Polynomial features add flexibility, but more features can also increase overfitting risk.
- Evaluate models on held-out data; compare metrics rather than assuming a more complex model performs better.

**Next steps:** try cross-validation, regularization (Ridge/Lasso), residual analysis, and a pipeline that keeps preprocessing inside each training fold.